In [1]:
import pandas as pd
import numpy as np


In [3]:
from google.colab import files
uploaded = files.upload()


df=pd.read_csv('labeled_data.csv')
df.isna().sum()

Saving labeled_data.csv to labeled_data.csv


,0
Unnamed: 0,0
count,0
hate_speech,0
offensive_language,0
neither,0
class,0
tweet,0


In [4]:
df.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [5]:
data=df[['tweet','class']]
data.head()

,tweet,class
0,!!! RT @mayasolovely: As a woman you shouldn't...,2
1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,1
2,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,1
3,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,1
4,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,1


In [6]:
import re

def clean(text):
    text=text.lower()
    text=re.sub(r'http\S+|www\S+','',text)
    text=re.sub(r'@\w+', '', text)
    text=re.sub(r'#\w+', '', text)
    text=re.sub(r'[^a-z\s]', '', text)
    text=re.sub(r'\s+', ' ', text).strip()
    return text

data['cleaned_tweet']=data['tweet'].apply(clean)

/tmp/ipython-input-643930702.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['cleaned_tweet']=data['tweet'].apply(clean)


In [7]:
data.head()

,tweet,class,cleaned_tweet
0,!!! RT @mayasolovely: As a woman you shouldn't...,2,rt as a woman you shouldnt complain about clea...
1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,1,rt boy dats coldtyga dwn bad for cuffin dat ho...
2,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,1,rt dawg rt you ever fuck a bitch and she start...
3,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,1,rt she look like a tranny
4,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,1,rt the shit you hear about me might be true or...


In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout



In [9]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['tweet'])

sequences = tokenizer.texts_to_sequences(df['tweet'])
padded_sequences = pad_sequences(sequences, maxlen=100)

In [10]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split( padded_sequences, df['class'], test_size=0.2, random_state=42)

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

num_classes = 3

model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=64, input_shape=(100,)))  # instead of input_length
model.add(LSTM(64))
model.add(Dropout(0.5))
model.add(Dense(3, activation='softmax'))  # 3 for multiclass

model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 673,219 (2.57 MB)

 Trainable params: 673,219 (2.57 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


In [13]:
print(np.unique(y_train))



[0 1 2]


In [14]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
x_train = np.array(x_train)
y_train = np.array(y_train)

model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)


Epoch 1/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8064 - loss: 0.5689 - val_accuracy: 0.9082 - val_loss: 0.2778
Epoch 2/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9210 - loss: 0.2352 - val_accuracy: 0.9047 - val_loss: 0.2669
Epoch 3/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9445 - loss: 0.1641 - val_accuracy: 0.9047 - val_loss: 0.3141
Epoch 4/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9617 - loss: 0.1139 - val_accuracy: 0.8865 - val_loss: 0.3824


In [16]:
import pickle
import os

# Create a directory to store model files
os.makedirs('saved_models', exist_ok=True)

print("Saving trained model and tokenizer...")

# Save the trained model
model.save('saved_models/hate_speech_model.keras')  # ✅ New format
print("✅ Model saved to: saved_models/hate_speech_model.keras")

# Alternative: Save as HDF5 format
model.save('saved_models/hate_speech_model.h5')
print("✅ Model also saved as: saved_models/hate_speech_model.h5")


# Save the tokenizer
with open('saved_models/tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("✅ Tokenizer saved to: saved_models/tokenizer.pickle")

# Save class mappings for reference
label_mapping = {
    0: 'Hate Speech',
    1: 'Offensive Language',
    2: 'Neither'
}

with open('saved_models/label_mapping.pickle', 'wb') as handle:
    pickle.dump(label_mapping, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("✅ Label mapping saved to: saved_models/label_mapping.pickle")

# Save model configuration info
model_info = {
    'vocab_size': 10000,
    'max_length': 100,
    'embedding_dim': 64,
    'lstm_units': 64,
    'num_classes': 3,
    'class_distribution': df['class'].value_counts().to_dict()
}

with open('saved_models/model_info.pickle', 'wb') as handle:
    pickle.dump(model_info, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("✅ Model info saved to: saved_models/model_info.pickle")

print("\n🎉 All model files saved successfully!")
print("📁 Files in saved_models/:")
for file in os.listdir('saved_models'):
    print(f"   - {file}")


def load_saved_model():
    """
    Load the saved model and tokenizer for making predictions
    """
    from tensorflow.keras.models import load_model
    import pickle

    loaded_model = load_model('saved_models/hate_speech_model.h5')
    print("✅ Model loaded successfully!")


    with open('saved_models/tokenizer.pickle', 'rb') as handle:
        loaded_tokenizer = pickle.load(handle)
    print("✅ Tokenizer loaded successfully!")

    with open('saved_models/label_mapping.pickle', 'rb') as handle:
        loaded_labels = pickle.load(handle)
    print("✅ Label mapping loaded successfully!")

    return loaded_model, loaded_tokenizer, loaded_labels



def predict_hate_speech_saved(text, model_path='saved_models/hate_speech_model.h5'):
    """
    Make predictions using saved model (no need to retrain)
    """
    from tensorflow.keras.models import load_model
    from tensorflow.keras.utils import pad_sequences
    import pickle
    import numpy as np

    # Load saved components
    model = load_model(model_path)

    with open('saved_models/tokenizer.pickle', 'rb') as handle:
        tokenizer = pickle.load(handle)

    with open('saved_models/label_mapping.pickle', 'rb') as handle:
        label_mapping = pickle.load(handle)

    # Preprocess the input text
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = pad_sequences(sequence, maxlen=100)

    # Make prediction
    prediction = model.predict(padded_sequence, verbose=0)
    predicted_class = np.argmax(prediction, axis=1)[0]
    confidence = float(np.max(prediction))

    # Return detailed results
    result = {
        'text': text,
        'predicted_class': int(predicted_class),
        'predicted_label': label_mapping[predicted_class],
        'confidence': round(confidence, 4),
        'all_probabilities': {
            'hate_speech': round(float(prediction[0][0]), 4),
            'offensive_language': round(float(prediction[0][1]), 4),
            'neither': round(float(prediction[0][2]), 4)
        }
    }

    return result


Saving trained model and tokenizer...
✅ Model saved to: saved_models/hate_speech_model.keras
✅ Model also saved as: saved_models/hate_speech_model.h5
✅ Tokenizer saved to: saved_models/tokenizer.pickle
✅ Label mapping saved to: saved_models/label_mapping.pickle
✅ Model info saved to: saved_models/model_info.pickle

🎉 All model files saved successfully!
📁 Files in saved_models/:
   - hate_speech_model.keras
   - tokenizer.pickle
   - label_mapping.pickle
   - hate_speech_model.h5
   - model_info.pickle


In [19]:
def interactive_prediction():
    """
    Interactive function to test the model with user input
    """
    print("🤖 Hate Speech Detection - Interactive Mode")
    print("Type 'quit' to exit\n")

    while True:
        user_input = input("Enter text to classify: ").strip()

        if user_input.lower() in ['quit', 'exit', 'q']:
            print("👋 Goodbye!")
            break

        if not user_input:
            print("⚠️  Please enter some text")
            continue

        try:
            result = predict_hate_speech_saved(user_input)
            print(f"\n📊 Results:")
            print(f"   Prediction: {result['predicted_label']}")
            print(f"   Confidence: {result['confidence']:.2%}")
            print(f"   Detailed probabilities:")
            for label, prob in result['all_probabilities'].items():
                print(f"     - {label.replace('_', ' ').title()}: {prob:.2%}")
            print()

        except Exception as e:
            print(f"❌ Error: {e}\n")

interactive_prediction()


🤖 Hate Speech Detection - Interactive Mode
Type 'quit' to exit

Enter text to classify: quit
👋 Goodbye!


In [ ]:
df['class'].value_counts()
